### Imports and data loading

In [10]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import torch.nn.functional as F
import random
from datasets import Dataset
from torch import nn
from torch.optim import AdamW

train_data = pd.read_csv("train_v3/train_data.csv")

raw_questions = list(pd.read_csv("train_v3/raw_questions.csv")['question'])

In [2]:
similar = train_data[['question1', 'question2']]
similar_q1 = list(similar['question1'])
similar_q2 = list(similar['question2'])

### Finetune

In [12]:
model = AutoModelForSequenceClassification.from_pretrained("train_v3//my_custom_bert_model", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained("train_v3//my_custom_bert_model")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: train_v3//my_custom_bert_model
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
tokenized_raw = tokenizer(raw_questions, return_tensors='pt', padding=True, truncation=True)
raw_emb = model(**tokenized_raw).last_hidden_state.mean(dim=1)
emb_norm = F.normalize(raw_emb, dim=1)
sim_matrix = emb_norm @ emb_norm.T

sim_matrix.fill_diagonal_(-float('inf'))
n = 1000
values, indices = torch.topk(sim_matrix.reshape(-1), k=n)

N = 1000
rows = indices // N
cols = indices % N
top_pairs = list(zip(rows.detach().numpy(), cols.detach().numpy(), values.detach().numpy()))

In [5]:
pos_pairs = [{'q1': q1, 'q2': q2, 'label': 1} for q1, q2 in zip(similar_q1, similar_q2)]
neg_pairs = [{'q1': raw_questions[top_pairs[i][0]], 'q2': raw_questions[top_pairs[i][1]], 'label': 0} for i in range(201, 701)]
all_pairs = pos_pairs + neg_pairs
random.shuffle(all_pairs)

In [6]:
ds = Dataset.from_list(all_pairs).train_test_split(test_size=0.1)
train_ds, val_ds = ds['train'], ds['train']

In [7]:
def tokenize(batch):
    return tokenizer(batch['q1'], batch['q2'], return_tensors='pt', padding=True, truncation=True)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['q1', 'q2'])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=['q1', 'q2'])

train_ds.set_format("torch")
val_ds.set_format("torch")

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(outputs):
    logits, labels = outputs
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [15]:
from transformers import TrainingArguments, Trainer
args = TrainingArguments(
    output_dir='./finetune_bert',
    num_train_epochs=25,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1'
)

trainer = Trainer(model, args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.359125,0.837778,0.855731
2,No log,0.254210,0.904444,0.911340
3,No log,0.189015,0.930000,0.934715
4,No log,0.098587,0.968889,0.969892
5,No log,0.083806,0.968889,0.969828
6,No log,0.039581,0.987778,0.987952
7,No log,0.015731,0.996667,0.996678
8,No log,0.008079,0.998889,0.998890
9,No log,0.005553,0.998889,0.998890
10,No log,0.007373,0.998889,0.998890


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [18]:
trainer.save_model("finetuned_bert_dupes")
tokenizer.save_pretrained("finetuned_bert_dupes")

from torch.nn.functional import softmax

def are_duplicates(q1: str, q2: str, threshold=0.5) -> bool:
    inputs = tokenizer(q1, q2, return_tensors="pt",
                       truncation=True, max_length=128)
    
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
    prob_duplicate = softmax(logits, dim=-1)[0][1].item()
    print(f"Duplicate probability: {prob_duplicate:.3f}")
    return prob_duplicate > threshold

are_duplicates("How do I reset my password?", "I forgot my password, how to recover it?")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Duplicate probability: 0.685


True

In [19]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.0012958558509126306, 'eval_accuracy': 1.0, 'eval_f1': 1.0}
